# NB3_SFT_NoPPL_Colab
HF project: `boods/FrMedQA-CrossLingual-v2-NoPPL-<qlora|bf16>-*`
Path: `/content/drive/MyDrive/00_PHD_THESIS/frmedqa-colab-v2`


In [ ]:
# ⚙️ Stage 0 — Install dependencies (Colab)
import subprocess, sys, importlib

REQUIRED = {"unsloth":"unsloth","transformers":"transformers","peft":"peft",
            "bitsandbytes":"bitsandbytes","datasets":"datasets",
            "rouge_score":"rouge-score","nltk":"nltk","bert_score":"bert-score",
            "huggingface_hub":"huggingface-hub","tqdm":"tqdm",
            "sklearn":"scikit-learn","sacrebleu":"sacrebleu", "modelscope": "modelscope",
            "datasketch": "datasketch"}

missing_pkgs_to_install = []
for mod, pkg in REQUIRED.items():
    if importlib.util.find_spec(mod) is None:
        missing_pkgs_to_install.append(pkg)

if missing_pkgs_to_install:
    print(f"Attempting to install missing packages: {missing_pkgs_to_install}")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing_pkgs_to_install)
        print("Successfully installed missing packages. A kernel restart is recommended for full effect.")
    except subprocess.CalledProcessError as e:
        print(f"Error during installation of {missing_pkgs_to_install}: {e}. Please install manually.")
        raise
    except Exception as e:
        print(f"An unexpected error occurred during package installation: {e}")
        raise

try:
    import nltk
    nltk.download("punkt_tab", quiet=True)
    nltk.download("punkt", quiet=True)
except Exception:
    pass
print("✓ Stage 0 done")


In [ ]:
# ⚙️ Stage 0b — Drive mount, paths, secrets
import os, sys

def _on_colab():
    try:
        import google.colab; return True
    except Exception: return False

if _on_colab():
    from google.colab import drive
    drive.mount("/content/drive")

    def _secret(name, prompt):
        try:
            from google.colab import userdata
            v = userdata.get(name)
            if v: return v
        except Exception: pass
        import getpass
        return getpass.getpass(prompt)
    os.environ["HF_TOKEN"] = _secret("HF_TOKEN", "HF_TOKEN: ")
    _wb = _secret("WANDB_API_KEY", "WANDB_API_KEY (blank to skip): ")
    if _wb:
        os.environ["WANDB_API_KEY"] = _wb
    else:
        os.environ["WANDB_DISABLED"] = "true"

BASE_DIR = os.environ.get("FRMEDQA_BASE", "/content/drive/MyDrive/00_PHD_THESIS/frmedqa-colab-v2")
REPO_DIR    = BASE_DIR
RESULTS_DIR = os.path.join(BASE_DIR, "RESULT")
EVALS_DIR   = os.path.join(BASE_DIR, "EVALS")
CACHE_DIR   = os.path.join(EVALS_DIR, "results_cache")
OUT_DIR     = os.path.join(BASE_DIR, "analysis_output")
FIG_DIR     = os.path.join(OUT_DIR, "figures")
for d in (RESULTS_DIR, EVALS_DIR, CACHE_DIR, OUT_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ["FRMEDQA_REPO"] = REPO_DIR
os.environ["FRMEDQA_BASE"] = BASE_DIR
os.environ["FRMEDQA_HF_USER"] = "boods"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

for v in ("HF_HOME", "HF_HUB_CACHE", "TRANSFORMERS_CACHE", "HUGGINGFACE_HUB_CACHE"):
    os.environ[v] = os.path.join(BASE_DIR, "hf_cache")

print(f"BASE_DIR: {BASE_DIR}")
print(f"CACHE_DIR: {CACHE_DIR}")


In [ ]:
os.environ["FRMEDQA_PROJECT_NAME"] = "FrMedQA-CrossLingual-v2-NoPPL"
if os.environ.get("FRMEDQA_SEED"):   # cluster multi-seed run: one HF namespace per seed
    os.environ["FRMEDQA_PROJECT_NAME"] += f"-s{os.environ['FRMEDQA_SEED']}"
print(f"HF project: {os.environ['FRMEDQA_PROJECT_NAME']}")

# ─── ONE precision for the whole project (single source of truth) ────────
# qlora = 4-bit NF4 frozen base + bf16 LoRA (QLoRA)   | fits L4 22 GB / A100 40 GB
# bf16  = bf16 frozen base + bf16 LoRA (16-bit LoRA) | needs A100 80 GB / H100
# To switch, edit PRECISION.txt on Drive; every notebook reads the same file.
_pf = os.path.join(BASE_DIR, "PRECISION.txt")
if not os.path.exists(_pf):
    open(_pf, "w").write("qlora")
os.environ["FRMEDQA_PRECISION"] = open(_pf).read().strip().lower()
assert os.environ["FRMEDQA_PRECISION"] in ("qlora", "bf16"), "PRECISION.txt must contain qlora or bf16"
print(f"Precision: {os.environ['FRMEDQA_PRECISION']}  (from {_pf})")


# NB3 — SFT NoPPL
Fine-tunes from `boods/FrMedQA-CrossLingual-v2-NoPPL-<qlora|bf16>-DAPT`.
Pushes to `boods/FrMedQA-CrossLingual-v2-NoPPL-<qlora|bf16>-{MCQA,ExtQA,AbsQA,Unified}`.

# 📓 NB3 — Multi-task Instruction Tuning (SFT) **EnToFrMedicaLLM pipeline · Notebook 3 of 6**
Loads the DAPT adapter from NB2 and trains task-specific LoRA adapters on the unified train/val splits NB1 produced. Output: four adapters pushed to HF Hub:
- `EnToFrMedicaLLM-MCQA` — multiple-choice QA only
- `EnToFrMedicaLLM-ExtQA` — extractive span QA only
- `EnToFrMedicaLLM-AbsQA` — open-ended QA only
- `EnToFrMedicaLLM-Unified` — all three tasks mixed (the headline model)### Design
- **Start from DAPT**: `cfg.hf_repo_dapt` is loaded as the base. SFT learns task formatting  on top of the domain-adapted weights (LoRA-on-LoRA via merging is the standard pattern).
- **r=32 / α=64** — same capacity as DAPT. Higher than DAPT often overfits small SFT data.
- **`packing=True`** — same multipack speedup.
- **Multi-seed**: train each adapter under `cfg.sft_seeds` (default `[42]`; expand to  `[42,43,44]` for variance estimates in the paper).
- **Crash-safe**: every adapter has its own `output_dir`; resumes from latest checkpoint.

## ⚙️ Stage 0 — Install

In [ ]:
# Stage 0+ — Suppress noisy deprecation/info messages
#
# Two sources of log spam dominate Colab cells:
#   1. transformers' AttentionMaskConverter deprecation warning fires on every
#      attention layer on every forward pass (40+ layers × every train step ×
#      every eval step = thousands of identical warnings).
#   2. Unsloth's "Restored added_tokens_decoder metadata" INFO line prints once
#      per saved checkpoint — fine in isolation, awful when checkpoints save
#      every 184 steps.
#
# We silence (1) via warnings filter and (2) via Python's logging level on
# Unsloth's logger. Neither affects training correctness; only output noise.
import warnings, logging

# (1) AttentionMaskConverter / modeling_attn_mask_utils deprecation
warnings.filterwarnings(
    "ignore",
    message=r".*attention mask API.*",
    category=FutureWarning,
)
warnings.filterwarnings(
    "ignore",
    message=r".*AttentionMaskConverter.*",
    category=FutureWarning,
)
warnings.filterwarnings(
    "ignore",
    message=r".*modeling_attn_mask_utils.*",
    category=FutureWarning,
)

# Belt-and-suspenders: silence the module's logger directly
for _name in ["transformers.modeling_attn_mask_utils",
              "transformers.masking_utils"]:
    logging.getLogger(_name).setLevel(logging.ERROR)

# (2) Unsloth checkpoint-restore chatter — INFO-level, raise to WARNING
for _name in ["unsloth", "unsloth.save", "unsloth.tokenizer_utils",
              "unsloth.chat_templates"]:
    logging.getLogger(_name).setLevel(logging.WARNING)

# Some Unsloth versions print directly with `print()` rather than logging.
# Monkey-patch builtin print to swallow that one specific line.
import builtins as _bi
_real_print = _bi.print
def _filtered_print(*args, **kwargs):
    if args and isinstance(args[0], str) and (
        "Restored added_tokens_decoder metadata" in args[0]
        or "attention mask API" in args[0]
    ):
        return  # swallow
    return _real_print(*args, **kwargs)
_bi.print = _filtered_print

print("✓ Stage 0+ done — warning suppressors active")

## 🔑 Stage 1 — Boilerplate / config / auth

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Stage 1 — import shared modules (cluster-ready)
#   Code (enmed_core / enmed_recipe) comes from the cloned repo + `poetry install`.
#   DATA & OUTPUTS live under $FRMEDQA_BASE (point it at scratch). Code != data.
import os, sys
BASE_DIR = os.environ.get("FRMEDQA_BASE", os.path.expanduser("~/frmedqa_experiments"))
def _find_repo_on_path():
    try:
        import enmed_core  # already importable (cwd = repo root, or installed)
        return
    except ModuleNotFoundError:
        for cand in [os.environ.get("FRMEDQA_REPO"), os.getcwd(),
                     os.path.abspath(os.path.join(os.getcwd(), ".."))]:
            if cand and os.path.exists(os.path.join(cand, "enmed_core.py")):
                sys.path.insert(0, cand); return
        raise ModuleNotFoundError("enmed_core not found — run `poetry install` in the repo, launch Jupyter from the repo root, or set $FRMEDQA_REPO to the repo path.")
_find_repo_on_path()
import enmed_core as ec
print(f"✓ enmed_core imported | BASE_DIR = {BASE_DIR}")

Mounted at /content/drive
✓ enmed_core imported


In [ ]:
# Stage 1b — Config / logger / auth
FULL = os.environ.get("FRMEDQA_PROFILE", "colab") == "full"   # cluster: full-scale settings
SEED = int(os.environ.get("FRMEDQA_SEED", "42"))
cfg = ec.EnMedConfig(
    base_dir=BASE_DIR, seed=SEED,
    sft_lr=2e-4, sft_lora_r=32, sft_lora_alpha=64, sft_lora_dropout=0.05,
    sft_epochs_mcqa=3, sft_epochs_extqa=3, sft_epochs_absqa=2, sft_epochs_unified=2,
    sft_batch_size=2, sft_grad_accum=(8 if FULL else 4), sft_seeds=[SEED],
    max_seq_length=(2048 if FULL else 1024),
)
cfg.make_dirs()
logger = ec.setup_logger("nb3",
    log_file=os.path.join(cfg.training_dir, f"nb3_sft_{ec.now_ts()}.log"))
ec.seed_all(cfg.seed)
ec.colab_keep_alive()

HF_TOKEN = ec.bootstrap_hf(logger=logger)
USE_WANDB = ec.bootstrap_wandb(project=cfg.wandb_project, logger=logger)

# Adapter output paths (one per task)
SFT_DIRS = {
    "mcqa":    os.path.join(cfg.training_dir, f"qwen3-14b-sft-mcqa-noppl-v2-{cfg.precision}"),
    "extqa":   os.path.join(cfg.training_dir, f"qwen3-14b-sft-extqa-noppl-v2-{cfg.precision}"),
    "absqa":   os.path.join(cfg.training_dir, f"qwen3-14b-sft-absqa-noppl-v2-{cfg.precision}"),
    "unified": os.path.join(cfg.training_dir, f"qwen3-14b-sft-unified-noppl-v2-{cfg.precision}"),
}
for d in SFT_DIRS.values(): os.makedirs(d, exist_ok=True)

HF_REPOS = {
    "mcqa":    f"{cfg.hf_user}/{cfg.project_name}-MCQA",
    "extqa":   f"{cfg.hf_user}/{cfg.project_name}-ExtQA",
    "absqa":   f"{cfg.hf_user}/{cfg.project_name}-AbsQA",
    "unified": cfg.hf_repo_unified,
}
cfg.save()
logger.info("=" * 70)
# Where NB2 Stage 16 saved the merged DAPT model (Drive first, Hub fallback).
# Defined here so this notebook works with any enmed_core.py version.
def _dapt_merged_source():
    sub = "qwen3-14b-cpt-noppl" if "noppl" in cfg.project_name.lower() else "qwen3-14b-cpt"
    local = os.path.join(cfg.training_dir, f"{sub}-v2-{cfg.precision}-merged-bf16")
    if os.path.exists(os.path.join(local, "config.json")):
        logger.info(f"DAPT-merged model: Drive folder {local}")
        return local
    logger.info(f"DAPT-merged model: Hub repo {cfg.hf_repo_dapt_merged} (no Drive folder at {local})")
    return cfg.hf_repo_dapt_merged

logger.info(f"NB3 — SFT  | DAPT base: {_dapt_merged_source()} | precision={cfg.precision}")
for k, d in SFT_DIRS.items():
    logger.info(f"           | {k:8s} → {HF_REPOS[k]}")
logger.info("=" * 70)

<IPython.core.display.Javascript object>

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


01:15:32 | I | ✓ HF authenticated


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: abel57583 (nlp237) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


01:15:38 | I | ✓ W&B authenticated (project=EnToFrMedicaLLM)
01:15:38 | I | ======================================================================
01:15:38 | I | NB3 — SFT  | DAPT base: boods/EnToFrMedicaLLM-DAPT
01:15:38 | I |            | mcqa     → boods/EnToFrMedicaLLM-MCQA
01:15:38 | I |            | extqa    → boods/EnToFrMedicaLLM-ExtQA
01:15:38 | I |            | absqa    → boods/EnToFrMedicaLLM-AbsQA
01:15:38 | I |            | unified  → boods/EnToFrMedicaLLM-Unified
01:15:38 | I | ======================================================================


## 📚 Stage 2 — Load NB1 splits + format per-task instructions

In [ ]:
# Stage 2a — Load train/val from NB1
TRAIN_PATH = os.path.join(cfg.results_dir, "train.jsonl")
VAL_PATH   = os.path.join(cfg.results_dir, "val.jsonl")
for p in (TRAIN_PATH, VAL_PATH):
    if not os.path.exists(p):
        raise FileNotFoundError(f"NB1 output missing: {p} — run NB1 first.")

train_rows = ec.load_jsonl(TRAIN_PATH)
val_rows   = ec.load_jsonl(VAL_PATH)
logger.info(f"  train={len(train_rows):,}  val={len(val_rows):,}")

# Per-task partitions
def _by_task(rows, task): return [r for r in rows if r["task"] == task]
TASK_TRAIN = {t: _by_task(train_rows, t) for t in ["mcqa","extqa","absqa"]}
TASK_VAL   = {t: _by_task(val_rows,   t) for t in ["mcqa","extqa","absqa"]}
for t in ["mcqa","extqa","absqa"]:
    logger.info(f"  {t:6s} | train={len(TASK_TRAIN[t]):,}  val={len(TASK_VAL[t]):,}")

01:15:41 | I |   train=15,766  val=1,155
01:15:41 | I |   mcqa   | train=1,969  val=311
01:15:41 | I |   extqa  | train=9,849  val=359
01:15:41 | I |   absqa  | train=3,948  val=485


In [ ]:
# Stage 2b — Convert each row → Qwen3-thinking conversation
def _opts_dict_or_list(opts):
    if isinstance(opts, dict): return opts
    if isinstance(opts, list): return {chr(65+i): str(o) for i,o in enumerate(opts)}
    return {}

def row_to_messages(r):
    """Build the supervised conversation for one row."""
    if r["task"] == "mcqa":
        msgs = ec.build_mcqa_messages(r["question"], _opts_dict_or_list(r["options"]))
    elif r["task"] == "extqa":
        msgs = ec.build_extqa_messages(r.get("context",""), r["question"])
    else:  # absqa
        msgs = ec.build_absqa_messages(r["question"], r.get("context"))
    msgs.append({"role": "assistant",
                 "content": r["answer"]})
    return msgs

## 🤖 Stage 3 — Reusable trainer factory

In [ ]:
# Stage 3a — Factory: load DAPT base + attach a fresh LoRA + return trainer
import math, torch
from datasets import Dataset as HFDataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from trl import SFTTrainer, SFTConfig
import multiprocessing as mp
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'
NUM_PROC = max(1, mp.cpu_count() - 1)

def build_train_pack(rows):
    """Rows → templated text dataset filtered to ≤ max_seq_length."""
    convos = [row_to_messages(r) for r in rows]
    ds = HFDataset.from_dict({"conversations": convos})
    def _apply(ex):
        out = []
        for c in ex["conversations"]:
            try:
                out.append(tokenizer.apply_chat_template(
                    c, tokenize=False, add_generation_prompt=False))
            except Exception:
                out.append(None)
        return {"text": out}
    ds = ds.map(_apply, batched=True, num_proc=NUM_PROC)
    ds = ds.filter(lambda x: x["text"] is not None and len(x["text"]) > 0)
    _tok = getattr(tokenizer, "tokenizer", tokenizer)
    def _len_ok(b):
        ids = _tok(b["text"], truncation=False, padding=False,
                   add_special_tokens=False)["input_ids"]
        return [len(x) <= cfg.max_seq_length for x in ids]
    ds = ds.filter(_len_ok, batched=True, num_proc=NUM_PROC)
    return ds


def make_trainer(train_ds, val_ds, output_dir, run_name, n_epochs, seed):
    total_steps = math.ceil(
        len(train_ds) / (cfg.sft_batch_size * cfg.sft_grad_accum)) * n_epochs
    warmup = max(1, int(total_steps * 0.05))
    save_steps = max(1, int(total_steps * cfg.save_every_n_steps_frac))

    args = SFTConfig(
        output_dir=output_dir,
        num_train_epochs=n_epochs,
        per_device_train_batch_size=cfg.sft_batch_size,
        gradient_accumulation_steps=cfg.sft_grad_accum,
        learning_rate=cfg.sft_lr,
        lr_scheduler_type="cosine",
        warmup_steps=warmup,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=max(1, total_steps // 100),
        save_strategy="steps", save_steps=save_steps,
        save_total_limit=cfg.keep_n_checkpoints,
        eval_strategy="steps", eval_steps=save_steps,
        report_to="wandb" if USE_WANDB else "none",
        run_name=run_name,
        seed=seed, gradient_checkpointing=True,
        optim="adamw_8bit", weight_decay=0.01, max_grad_norm=1.0,
        max_seq_length=cfg.max_seq_length,
        dataset_text_field="text",
        packing=True, push_to_hub=False,
    )
    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer,
        train_dataset=train_ds, eval_dataset=val_ds, args=args,
    )
    return train_on_responses_only(trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
# Stage 3b — Load the DAPT-merged 16-bit checkpoint at cfg.precision.
#   qlora → quantized on load to 4-bit NF4, frozen (QLoRA)
#   bf16  → kept in bf16, frozen (16-bit LoRA)
# The DAPT LoRA was merged into the bf16 base in NB2 (Stage 16), so no adapter
# is ever merged into 4-bit weights here.
import gc, torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

def load_dapt_base():
    """(Re)load the clean DAPT base. Called once per task so every task
    adapter is trained on exactly the same weights (DAPT only)."""
    global model, tokenizer
    for _n in ("trainer", "model"):
        if _n in globals(): del globals()[_n]
    gc.collect(); torch.cuda.empty_cache()
    model, tokenizer = ec.load_model(cfg, _dapt_merged_source(), logger=logger)
    tokenizer = get_chat_template(tokenizer, chat_template="qwen3")
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return model, tokenizer

ec.check_vram_for_precision(cfg, logger=logger)
model, tokenizer = load_dapt_base()
logger.info(f"  ✓ DAPT base ready | vocab={len(tokenizer):,}")


01:20:42 | I | Loading DAPT base: boods/EnToFrMedicaLLM-DAPT
==((====))==  Unsloth 2026.4.8: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


## 🟦 Stage 4 — Train the four adapters

In [ ]:
# Stage 4 — Train each task adapter; multi-seed if cfg.sft_seeds has >1 entry
def attach_fresh_lora(seed):
    """Attach a brand-new task LoRA to a CLEAN DAPT base.

    If a previous task left its adapter on the model, the base is reloaded
    instead of merging that adapter in, so task adapters never leak into each
    other: MCQA, ExtQA, AbsQA and Unified are each trained on DAPT only,
    which is exactly how NB5b evaluates them.
    """
    global model, tokenizer
    if hasattr(model, "peft_config"):
        model, tokenizer = load_dapt_base()
    return FastLanguageModel.get_peft_model(
        model,
        r=cfg.sft_lora_r, lora_alpha=cfg.sft_lora_alpha,
        lora_dropout=cfg.sft_lora_dropout,
        target_modules=cfg.sft_lora_targets, bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=seed, use_rslora=False, loftq_config=None,
    )


def train_one(task, train_rows, val_rows, n_epochs):
    """Train one adapter (one seed) on a subset of rows."""
    global model
    seed = cfg.sft_seeds[0]
    out_dir = SFT_DIRS[task]
    run = f"SFT_{task}_seed{seed}_{ec.now_ts()}"
    logger.info(f"=== {task.upper()} | rows={len(train_rows):,} | epochs={n_epochs} | seed={seed} ===")

    # Strip any prior task adapter, then attach a fresh one
    model = attach_fresh_lora(seed)
    model.print_trainable_parameters()

    train_ds = build_train_pack(train_rows)
    val_ds   = build_train_pack(val_rows)
    trainer = make_trainer(train_ds, val_ds, out_dir, run, n_epochs, seed)

    result = ec.resumable_train(trainer, out_dir, auto_resume=cfg.auto_resume, logger=logger)
    logger.info(f"  {task} | train_loss={result.training_loss:.4f}")

    # Save + push
    model.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)
    ec.drive_flush()

    if HF_TOKEN:
        try:
            model.push_to_hub(HF_REPOS[task], token=HF_TOKEN)
            tokenizer.push_to_hub(HF_REPOS[task], token=HF_TOKEN)
            logger.info(f"  ✓ pushed → https://huggingface.co/{HF_REPOS[task]}")
        except Exception as e:
            logger.error(f"  push failed: {e}")

    metrics = {
        "task": task, "seed": int(seed), "n_epochs": int(n_epochs),
        "train_loss": float(result.training_loss),
        "n_train": len(train_rows), "n_val": len(val_rows),
    }
    ec.atomic_write_json(metrics, os.path.join(cfg.results_dir, f"sft_{task}_metrics.json"))
    return metrics

In [ ]:
# Stage 4 — Train selected task adapter(s). PARALLEL-JOB FRIENDLY.
# Choose tasks via env FRMEDQA_SFT_TASKS (comma-separated). Default = all four,
# so running this notebook normally (local debug) trains everything. The SLURM
# job array (scripts/sft_array.sbatch) sets ONE task per GPU so all run at once.
import os as _os, random as _r
_ALL = ["mcqa", "extqa", "absqa", "unified"]
_sel = _os.environ.get("FRMEDQA_SFT_TASKS", "").strip()
TASKS_TO_RUN = [t.strip() for t in _sel.split(",") if t.strip()] if _sel else list(_ALL)
for _t in TASKS_TO_RUN:
    if _t not in _ALL:
        raise ValueError(f"Unknown SFT task {_t!r}. Valid: {_ALL}")
logger.info(f"SFT tasks for this job: {TASKS_TO_RUN}")

def _task_data(task):
    if task == "unified":
        tr = TASK_TRAIN["mcqa"] + TASK_TRAIN["extqa"] + TASK_TRAIN["absqa"]
        vl = TASK_VAL["mcqa"]   + TASK_VAL["extqa"]   + TASK_VAL["absqa"]
        _r.Random(cfg.seed).shuffle(tr)
        return tr, vl, cfg.sft_epochs_unified
    ep = {"mcqa": cfg.sft_epochs_mcqa, "extqa": cfg.sft_epochs_extqa,
          "absqa": cfg.sft_epochs_absqa}[task]
    return TASK_TRAIN[task], TASK_VAL[task], ep

RESULTS = {}
for task in TASKS_TO_RUN:
    _tr, _vl, _ep = _task_data(task)
    RESULTS[task] = train_one(task, _tr, _vl, n_epochs=_ep)
    ec.cleanup()

## 📊 Stage 5 — Loss summary

In [15]:
# Stage 5 — Loss summary (handles any subset of tasks)
import pandas as pd
df = pd.DataFrame(list(RESULTS.values()))
print(df[["task","n_train","n_val","n_epochs","train_loss"]].to_string(index=False))
# Single-task (parallel) runs write a per-task file so concurrent jobs don't clobber.
_suffix = "" if set(RESULTS) == set(_ALL) else "_" + "_".join(sorted(RESULTS))
ec.atomic_write_json(df.to_dict(orient="records"),
                     os.path.join(cfg.results_dir, f"sft_summary{_suffix}.json"))

   task  n_train  n_val  n_epochs  train_loss
   mcqa     1969    311         3    0.272498
  extqa     9849    359         3    0.133126
  absqa     3948    485         2    1.442275
unified    15766   1155         2    0.914700


'/content/drive/MyDrive/00_PHD_THESIS/CODE/05-05-26/RESULT/sft_summary.json'

## 🏁 Stage 6 — Done. Open NB4 (Export & Quantization).

In [16]:
print("\u2550" * 70)
print(f"  NB3 (SFT) COMPLETE — {ec.now_ts()}  | tasks: {list(RESULTS)}")
print("\u2550" * 70)
for _t, _m in RESULTS.items():
    print(f"  {_t:8s} loss={_m['train_loss']:.4f}  \u2192  {HF_REPOS[_t]}")
print("\u2550" * 70)
print("  \u2192 Open NB4_ExportQuant.ipynb")

══════════════════════════════════════════════════════════════════════
  NB3 (SFT) COMPLETE — 20260506_192020
══════════════════════════════════════════════════════════════════════
  mcqa     loss=0.2725  →  boods/EnToFrMedicaLLM-MCQA
  extqa    loss=0.1331  →  boods/EnToFrMedicaLLM-ExtQA
  absqa    loss=1.4423  →  boods/EnToFrMedicaLLM-AbsQA
  unified  loss=0.9147  →  boods/EnToFrMedicaLLM-Unified
══════════════════════════════════════════════════════════════════════
  → Open NB4_ExportQuant.ipynb
